# Import Packages

In [1]:
import os
import numpy as np
from bci_essentials.io.xdf_sources import XdfEegSource, XdfMarkerSource
from bci_essentials.bci_controller import BciController
from bci_essentials.paradigm.ssvep_paradigm import SsvepParadigm
from bci_essentials.data_tank.data_tank import DataTank
from bci_essentials.utils.logger import Logger  # Logger wrapper
from bci_essentials.classification.ssvep_fbcca_classifier import (
    SsvepFbCcaClassifier,
)

# Import Data & Initialize Objects

In [2]:
filename = os.path.join("data", "sub-Eli_ses-S003_task-T1_run-001_eeg.xdf")

logger = Logger(name=__name__)
logger.setLevel(1)
eeg_source = XdfEegSource(filename)
marker_source = XdfMarkerSource(filename)
paradigm = SsvepParadigm(live_update=False, iterative_training=False) # Setting live_upadate to False means it will classify each trial rather than each epoch (full 5 seconds rather than every second 5 time)
data_tank = DataTank()

2025-04-03 15:21:43 - INFO - __main__ : Setting logging level to Unknown Level


# Define the classifier

In [3]:
# Settings
target_frequencies = np.array([7.692307, 10, 12.5, 14.28571])
n_harmonics = 2  # Number of harmonics to use for SSVEP classification
fb_cutoffs = np.array([[i, 27] for i in range(5, 25, 2)])   # Filter bank cut-off frequencies
nsamples = 5 * 256  # 5 seconds of data at 256 Hz
fsample = 256.0  # Hz
concatenate_trials = True  # Concatenate trials for training

classifier = SsvepFbCcaClassifier(subset=[])
classifier.set_ssvep_settings(
    fsample=256.0,
    n_harmonics=n_harmonics,  # Number of harmonics to use for SSVEP classification
    target_freqs=target_frequencies,
    # filter_bank=fb_cutoffs,
    concatenate_trials=concatenate_trials,
)

2025-04-03 15:21:44 - INFO - bci_essentials.classification.generic_classifier : Initializing the classifier
2025-04-03 15:21:44 - WARNING - bci_essentials.signal_processing : SSVEP templates not computed because n_samples is None.


# Set up BCIController

In [4]:
test_ssvep = BciController(classifier, eeg_source, marker_source, paradigm, data_tank)

test_ssvep.setup(
    online=False,
    train_complete=True,  # Set to True to run the classifier on the entire dataset
)

2025-04-03 15:21:44 - INFO - bci_essentials.bci_controller : g.USBamp-1
2025-04-03 15:21:44 - INFO - bci_essentials.bci_controller : ['Fz', 'F4', 'F8', 'C3', 'Cz', 'C4', 'T8', 'P3', 'P4', 'P7', 'PO7', 'P8', 'PO8', 'O1', 'Oz', 'O2']


# Run

In [5]:
test_ssvep.run()

2025-04-03 15:21:44 - INFO - bci_essentials.bci_controller : Processed Marker: Cue Object : 2
2025-04-03 15:21:44 - INFO - bci_essentials.bci_controller : Processed Marker: Trial Started
2025-04-03 15:21:44 - INFO - bci_essentials.bci_controller : Processed Marker: ssvep,4,-1,1,7.692307,10,12.5,14.28571
2025-04-03 15:21:44 - INFO - bci_essentials.bci_controller : Processed Marker: ssvep,4,-1,1,7.692307,10,12.5,14.28571
2025-04-03 15:21:44 - INFO - bci_essentials.bci_controller : Processed Marker: ssvep,4,-1,1,7.692307,10,12.5,14.28571
2025-04-03 15:21:44 - INFO - bci_essentials.bci_controller : Processed Marker: ssvep,4,-1,1,7.692307,10,12.5,14.28571
2025-04-03 15:21:44 - INFO - bci_essentials.bci_controller : Processed Marker: ssvep,4,-1,1,7.692307,10,12.5,14.28571
2025-04-03 15:21:44 - WARNING - bci_essentials.classification.ssvep_fbcca_classifier : SSVEP templates created: (4, 6, 1280). If this occurs repeatedly, it may cause performance issues.
2025-04-03 15:21:44 - INFO - bci_esse

# Get targets

In [6]:
markers = test_ssvep.marker_data
targets = []

for marker in markers:
    if "Cue " in marker:
        targets.append(int(marker.split(" ")[-1]))

print(targets)

#remove last target (this is just because I stopped Eli's data collectoin in the middle of a trial so there is a cue for the last trial but no actual data/markers from the trial)
targets.pop()
print(targets)


[2, 1, 2, 1, 2, 3, 1, 2, 0, 0, 3, 0, 2, 1, 0, 0, 1, 1, 0, 3, 3, 3, 1, 1, 3, 1, 2, 1, 3, 3, 1, 2, 3, 0, 3, 3, 3, 3, 2, 3, 3, 1, 1, 2, 1, 2, 0, 0, 2, 1, 0, 0, 1, 0, 0, 3, 0, 1, 2, 1, 2, 0, 1, 0, 2, 1, 2, 0, 1, 3, 3, 0, 0, 2, 3, 0, 3, 1, 2, 0, 2, 0, 2, 3, 0, 3, 1, 0]
[2, 1, 2, 1, 2, 3, 1, 2, 0, 0, 3, 0, 2, 1, 0, 0, 1, 1, 0, 3, 3, 3, 1, 1, 3, 1, 2, 1, 3, 3, 1, 2, 3, 0, 3, 3, 3, 3, 2, 3, 3, 1, 1, 2, 1, 2, 0, 0, 2, 1, 0, 0, 1, 0, 0, 3, 0, 1, 2, 1, 2, 0, 1, 0, 2, 1, 2, 0, 1, 3, 3, 0, 0, 2, 3, 0, 3, 1, 2, 0, 2, 0, 2, 3, 0, 3, 1]
